# Stage 3: Graph Embedding Recommendation Experiments

This notebook continues **Stage 2** by moving from simple graph-overlap heuristics to **walk-based graph embedding methods**.

Goals for this notebook:
- load the existing SQLite dataset directly
- build a weighted **manga-to-manga projected graph** from shared metadata
- learn embeddings with **DeepWalk** and **Node2Vec**
- compare both models against the current **v3 content-based recommender**
- inspect example recommendations for one user before deciding on the next experiment


## Portability Notes

This notebook intentionally stays focused on experiments rather than Flask or web-app integration.

It only needs:
- the repo directory
- `data/db/manga.db`
- Python packages for data work: `pandas`, `numpy`, `scikit-learn`
- optionally `gensim` for skip-gram training

If `gensim` is unavailable, the notebook falls back to a **walk co-occurrence + TruncatedSVD** approximation so the workflow can still run on a lighter environment.


In [1]:
from pathlib import Path
import ast
import math
import random
import sqlite3
import sys
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

NOTEBOOK_DIR = Path.cwd().resolve()
candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
candidate_roots.extend([NOTEBOOK_DIR / 'shelf', *[parent / 'shelf' for parent in NOTEBOOK_DIR.parents]])

seen = set()
REPO_ROOT = None
for candidate in candidate_roots:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'data' / 'db' / 'manga.db').exists():
        REPO_ROOT = candidate
        break

assert REPO_ROOT is not None, 'Could not locate repo root containing data/db/manga.db'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from recommender.recommender import recommendation_scores

DB_PATH = REPO_ROOT / 'data' / 'db' / 'manga.db'
EXAMPLE_USER_ID = 'avreylavelle'
RNG_SEED = 42

print(f'Repo root: {REPO_ROOT}')
print(f'Database path: {DB_PATH}')
print(f'Database size (MB): {DB_PATH.stat().st_size / 1024**2:.1f}')
print(f'Example user for previews: {EXAMPLE_USER_ID}')


Repo root: /opt/shelf
Database path: /opt/shelf/data/db/manga.db
Database size (MB): 202.8
Example user for previews: avreylavelle


## Dataset Load

For stage 3 we keep the same general SQLite-first workflow from stage 2, but we restrict the item universe to titles with a **MAL mapping**.

That keeps the candidate set closer to the titles that already have baseline statistics available, which makes the `v3` comparison cleaner and keeps the graph smaller for walk-based experiments.


In [2]:
with sqlite3.connect(DB_PATH) as conn:
    users_df = pd.read_sql_query(
        'SELECT username, age, gender, language FROM users',
        conn,
    )
    ratings_df = pd.read_sql_query('SELECT * FROM user_ratings', conn)
    reading_df = pd.read_sql_query('SELECT * FROM user_reading_list', conn)
    dnr_df = pd.read_sql_query('SELECT * FROM user_dnr', conn)
    manga_df = pd.read_sql_query(
        '''
        SELECT
            core.id AS id,
            core.title_name,
            core.english_name,
            core.japanese_name,
            core.item_type,
            core.authors,
            core.genres,
            core.themes,
            core.status,
            core.publishing_date,
            map.mal_id,
            stats.score,
            stats.members,
            stats.popularity,
            stats.favorited
        FROM manga_core AS core
        INNER JOIN manga_map AS map ON map.mangadex_id = core.id
        LEFT JOIN manga_stats AS stats ON stats.mal_id = map.mal_id
        ''',
        conn,
    )

print('users_df:', users_df.shape)
print('ratings_df:', ratings_df.shape)
print('reading_df:', reading_df.shape)
print('dnr_df:', dnr_df.shape)
print('manga_df:', manga_df.shape)


users_df: (8, 4)
ratings_df: (87, 9)
reading_df: (20, 7)
dnr_df: (6, 6)
manga_df: (42532, 15)


In [3]:
def parse_literal_list(value):
    # Parse list-like text fields into normalized Python lists.
    if value is None:
        return []
    if isinstance(value, list):
        return [
            str(item).strip()
            for item in value
            if str(item).strip() and str(item).strip().lower() != 'none'
        ]

    text = str(value).strip()
    if not text or text.lower() in {'none', 'nan'} or text == '[]':
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [
                str(item).strip()
                for item in parsed
                if str(item).strip() and str(item).strip().lower() != 'none'
            ]
    except Exception:
        pass

    return [part.strip() for part in text.split(',') if part.strip()]


def pick_item_id(row):
    # Choose the most normalized manga identifier available in interaction tables.
    for column in ('canonical_id', 'mdex_id', 'manga_id'):
        value = row.get(column)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return None


ratings_edges = ratings_df.copy()
ratings_edges['user_id'] = ratings_edges['user_id'].astype(str).str.strip().str.lower()
ratings_edges['item_id'] = ratings_edges.apply(pick_item_id, axis=1)
ratings_edges['rating'] = pd.to_numeric(ratings_edges['rating'], errors='coerce')
ratings_edges = ratings_edges.dropna(subset=['item_id'])

reading_edges = reading_df.copy()
reading_edges['user_id'] = reading_edges['user_id'].astype(str).str.strip().str.lower()
reading_edges['item_id'] = reading_edges.apply(pick_item_id, axis=1)
reading_edges['status'] = reading_edges['status'].fillna('Plan to Read')
reading_edges = reading_edges.dropna(subset=['item_id'])

dnr_edges = dnr_df.copy()
dnr_edges['user_id'] = dnr_edges['user_id'].astype(str).str.strip().str.lower()
dnr_edges['item_id'] = dnr_edges.apply(pick_item_id, axis=1)
dnr_edges = dnr_edges.dropna(subset=['item_id'])

for column in ('genres', 'themes', 'authors'):
    manga_df[column] = manga_df[column].apply(parse_literal_list)
    manga_df[f'{column}_set'] = manga_df[column].apply(set)

manga_df['score'] = pd.to_numeric(manga_df['score'], errors='coerce').fillna(0.0)
manga_df['members'] = pd.to_numeric(manga_df['members'], errors='coerce').fillna(0.0)
manga_df['popularity'] = pd.to_numeric(manga_df['popularity'], errors='coerce')
manga_df = manga_df.drop_duplicates(subset=['id']).set_index('id', drop=False)

graph_ids = set(manga_df.index)

ratings_edges = ratings_edges[ratings_edges['item_id'].isin(graph_ids)].copy()
reading_edges = reading_edges[reading_edges['item_id'].isin(graph_ids)].copy()
dnr_edges = dnr_edges[dnr_edges['item_id'].isin(graph_ids)].copy()

interaction_summary = pd.DataFrame(
    [
        {'frame': 'ratings_edges', 'rows': len(ratings_edges), 'users': ratings_edges['user_id'].nunique()},
        {'frame': 'reading_edges', 'rows': len(reading_edges), 'users': reading_edges['user_id'].nunique()},
        {'frame': 'dnr_edges', 'rows': len(dnr_edges), 'users': dnr_edges['user_id'].nunique()},
        {'frame': 'manga_df', 'rows': len(manga_df), 'users': np.nan},
    ]
)

display(interaction_summary)
display(manga_df[['title_name', 'english_name', 'item_type', 'score', 'members']].head(5))


,frame,rows,users
0,ratings_edges,87,5.0
1,reading_edges,20,3.0
2,dnr_edges,6,2.0
3,manga_df,42532,NaN


,title_name,english_name,item_type,score,members
id,,,,,
0c040315-355c-4a38-b586-d7ad021ff914,Tonari no Danshi,Tonari no Danshi,Manga,7.15,4735.0
cedad717-2c62-4b7d-9963-ce2d3a16ca0b,Lovely Lesson - Hair Make-Up Hen,Lovely Lesson - Hair Make-Up Hen,Manga,6.88,386.0
15b9e249-57fd-460e-9c1e-0bd72e9d7af5,The IT Revolution,The IT Revolution,Manga,5.05,302.0
b3c0e0ea-7a1b-4602-a7a9-3ca078a13ad8,Amnesia Later,Amnesia Later,Manga,6.26,1709.0
aef414c1-702b-4583-b53e-916c44461462,Hidarite ga Yondeiru,Hidarite ga Yondeiru,Manga,0.00,79.0


## Stage 3 Graph Construction

Rather than using a full heterogeneous graph directly, this notebook builds a **projected manga-to-manga graph**.

Two manga are connected when they share metadata such as:
- genres
- themes
- authors

The projected graph is weighted so that:
- authors contribute more than themes or genres
- very common tags are downweighted with an inverse-frequency style term
- extremely common tags can be skipped entirely so the graph does not become dominated by generic metadata like `Romance`
- each manga keeps only its top weighted neighbors


In [4]:
GRAPH_CONFIG = {
    'relation_weights': {
        'genres': 1.0,
        'themes': 1.5,
        'authors': 3.0,
    },
    'max_relation_frequency': {
        'genres': 4000,
        'themes': 2500,
        'authors': 300,
    },
    'max_neighbors': 40,
}


def build_relation_index(frame, column):
    index = defaultdict(list)
    for row in frame.itertuples(index=False):
        for value in getattr(row, column):
            index[value].append(row.id)
    return index


relation_indices = {
    relation: build_relation_index(manga_df.reset_index(drop=True), relation)
    for relation in ('genres', 'themes', 'authors')
}
relation_degrees = {
    relation: {key: len(value) for key, value in index.items()}
    for relation, index in relation_indices.items()
}


def build_weighted_item_graph(
    frame,
    relation_indices,
    relation_weights,
    max_relation_frequency,
    max_neighbors=40,
):
    adjacency = {}
    started = time.time()

    for row in frame.itertuples(index=False):
        neighbor_scores = defaultdict(float)

        for relation, relation_weight in relation_weights.items():
            tags = getattr(row, relation)
            for tag in tags:
                neighbors = relation_indices[relation].get(tag, [])
                degree = len(neighbors)
                if degree <= 1:
                    continue
                if degree > max_relation_frequency[relation]:
                    continue

                increment = relation_weight / math.log1p(degree)
                for neighbor in neighbors:
                    if neighbor == row.id:
                        continue
                    neighbor_scores[neighbor] += increment

        ranked_neighbors = sorted(
            neighbor_scores.items(),
            key=lambda pair: (-pair[1], pair[0]),
        )[:max_neighbors]
        adjacency[row.id] = ranked_neighbors

    elapsed = time.time() - started
    return adjacency, elapsed


adjacency, build_seconds = build_weighted_item_graph(
    manga_df.reset_index(drop=True),
    relation_indices=relation_indices,
    relation_weights=GRAPH_CONFIG['relation_weights'],
    max_relation_frequency=GRAPH_CONFIG['max_relation_frequency'],
    max_neighbors=GRAPH_CONFIG['max_neighbors'],
)
adjacency_sets = {node: {neighbor for neighbor, _ in neighbors} for node, neighbors in adjacency.items()}

degree_series = pd.Series({node: len(neighbors) for node, neighbors in adjacency.items()}, name='degree')
graph_summary = pd.DataFrame(
    {
        'node_count': [len(adjacency)],
        'directed_edge_count': [int(degree_series.sum())],
        'avg_degree': [float(degree_series.mean())],
        'median_degree': [float(degree_series.median())],
        'isolated_nodes': [int((degree_series == 0).sum())],
        'build_seconds': [round(build_seconds, 2)],
    }
)

display(graph_summary)
display(degree_series.describe().to_frame())


,node_count,directed_edge_count,avg_degree,median_degree,isolated_nodes,build_seconds
0,42532,1139610,26.794179,40.0,2671,154.3


,degree
count,42532.000000
mean,26.794179
std,16.994563
min,0.000000
25%,6.000000
50%,40.000000
75%,40.000000
max,40.000000


## Walk-Based Embedding Methods

We train two walk-based models over the same weighted projected graph:

- **DeepWalk**: uniform random walks over the graph
- **Node2Vec**: biased second-order walks with `p` and `q`

Both methods use the same downstream recipe:
1. generate random walks
2. learn embeddings from walk co-occurrence
3. represent a user by averaging the embeddings of their positive seed manga
4. rank candidate manga by cosine similarity to that user vector


In [5]:
def weighted_choice(weighted_neighbors, rng):
    total = sum(weight for _, weight in weighted_neighbors)
    if total <= 0:
        return None
    threshold = rng.random() * total
    running = 0.0
    for node, weight in weighted_neighbors:
        running += weight
        if running >= threshold:
            return node
    return weighted_neighbors[-1][0]


def deepwalk_walk(start_node, adjacency, walk_length, rng):
    walk = [start_node]
    while len(walk) < walk_length:
        neighbors = adjacency.get(walk[-1], [])
        if not neighbors:
            break
        next_node = weighted_choice(neighbors, rng)
        if next_node is None:
            break
        walk.append(next_node)
    return walk


def node2vec_walk(start_node, adjacency, adjacency_sets, walk_length, rng, p=1.0, q=2.0):
    walk = [start_node]

    while len(walk) < walk_length:
        current = walk[-1]
        neighbors = adjacency.get(current, [])
        if not neighbors:
            break

        if len(walk) == 1:
            next_node = weighted_choice(neighbors, rng)
            if next_node is None:
                break
            walk.append(next_node)
            continue

        previous = walk[-2]
        previous_neighbors = adjacency_sets.get(previous, set())
        biased_neighbors = []

        for neighbor, base_weight in neighbors:
            if neighbor == previous:
                bias = 1.0 / p if p else 0.0
            elif neighbor in previous_neighbors:
                bias = 1.0
            else:
                bias = 1.0 / q if q else 0.0
            biased_neighbors.append((neighbor, base_weight * bias))

        next_node = weighted_choice(biased_neighbors, rng)
        if next_node is None:
            break
        walk.append(next_node)

    return walk


def generate_walks(
    nodes,
    adjacency,
    walk_length=20,
    num_walks=6,
    seed=42,
    method='deepwalk',
    adjacency_sets=None,
    p=1.0,
    q=2.0,
):
    rng = random.Random(seed)
    walk_nodes = [node for node in nodes if adjacency.get(node)]
    walks = []

    for _ in range(num_walks):
        rng.shuffle(walk_nodes)
        for node in walk_nodes:
            if method == 'node2vec':
                walk = node2vec_walk(
                    node,
                    adjacency=adjacency,
                    adjacency_sets=adjacency_sets,
                    walk_length=walk_length,
                    rng=rng,
                    p=p,
                    q=q,
                )
            else:
                walk = deepwalk_walk(node, adjacency=adjacency, walk_length=walk_length, rng=rng)
            if len(walk) > 1:
                walks.append(walk)

    return walks


In [6]:
def train_embeddings_from_walks(
    walks,
    dimensions=64,
    window=6,
    min_count=0,
    seed=42,
    workers=1,
):
    backend = None
    embeddings = {}

    try:
        from gensim.models import Word2Vec

        model = Word2Vec(
            sentences=walks,
            vector_size=dimensions,
            window=window,
            min_count=min_count,
            sg=1,
            workers=workers,
            seed=seed,
            epochs=5,
        )
        embeddings = {token: model.wv[token] for token in model.wv.index_to_key}
        backend = 'gensim_word2vec'

    except ModuleNotFoundError:
        from sklearn.decomposition import TruncatedSVD
        from sklearn.feature_extraction import DictVectorizer

        cooccurrence = defaultdict(Counter)
        for walk in walks:
            local_walk_length = len(walk)
            for idx, token in enumerate(walk):
                left = max(0, idx - window)
                right = min(local_walk_length, idx + window + 1)
                for neighbor_idx in range(left, right):
                    if idx == neighbor_idx:
                        continue
                    neighbor = walk[neighbor_idx]
                    distance = abs(idx - neighbor_idx)
                    if distance <= 0:
                        continue
                    cooccurrence[token][neighbor] += 1.0 / distance

        row_dicts = []
        tokens = []
        for token, counts in cooccurrence.items():
            row_dicts.append({f'neighbor::{neighbor}': math.log1p(weight) for neighbor, weight in counts.items()})
            tokens.append(token)

        vectorizer = DictVectorizer(sparse=True)
        matrix = vectorizer.fit_transform(row_dicts)
        max_components = min(dimensions, max(2, min(matrix.shape) - 1))
        svd = TruncatedSVD(n_components=max_components, random_state=seed)
        dense = svd.fit_transform(matrix)

        if dense.shape[1] < dimensions:
            padding = np.zeros((dense.shape[0], dimensions - dense.shape[1]), dtype=float)
            dense = np.hstack([dense, padding])

        embeddings = {token: vector for token, vector in zip(tokens, dense)}
        backend = 'cooccurrence_svd_fallback'

    normalized_embeddings = {}
    for token, vector in embeddings.items():
        vector = np.asarray(vector, dtype=float)
        norm = np.linalg.norm(vector)
        if norm > 0:
            normalized_embeddings[token] = vector / norm

    return normalized_embeddings, backend


def score_candidates_from_embeddings(seed_items, candidate_items, embeddings):
    seed_vectors = [embeddings[item] for item in seed_items if item in embeddings]
    if not seed_vectors:
        return pd.DataFrame(columns=['id', 'graph_score'])

    user_vector = np.mean(np.vstack(seed_vectors), axis=0)
    norm = np.linalg.norm(user_vector)
    if norm <= 0:
        return pd.DataFrame(columns=['id', 'graph_score'])
    user_vector = user_vector / norm

    rows = []
    for item in candidate_items:
        vector = embeddings.get(item)
        if vector is None:
            continue
        rows.append({'id': item, 'graph_score': float(np.dot(user_vector, vector))})

    if not rows:
        return pd.DataFrame(columns=['id', 'graph_score'])

    return pd.DataFrame(rows).sort_values('graph_score', ascending=False).reset_index(drop=True)


## Offline Evaluation Setup

The evaluation here is still intentionally lightweight because the user data is small.

We use the following setup:
- **relevant items** = positive ratings (`rating >= 7`)
- **seed items** = remaining positive ratings plus any `In Progress` reading-list items
- **blocked items** = all rated items, reading-list items, and DNR items, except the held-out test positives

The `v3` comparison is reconstructed **offline** using the same `recommendation_scores(..., mode='v3')` function as the application, but with a temporary seed profile built only from the train-side items in the split.


In [7]:
def build_user_records(ratings_edges, reading_edges, dnr_edges, valid_ids, positive_threshold=7.0):
    user_ids = sorted(
        set(ratings_edges['user_id'])
        | set(reading_edges['user_id'])
        | set(dnr_edges['user_id'])
    )
    records = {}

    for user_id in user_ids:
        user_ratings = ratings_edges[ratings_edges['user_id'] == user_id].copy()
        rating_map = {
            row.item_id: float(row.rating)
            for row in user_ratings.itertuples(index=False)
            if row.item_id in valid_ids and pd.notna(row.rating)
        }
        positive_ratings = {
            item_id
            for item_id, rating in rating_map.items()
            if rating >= positive_threshold
        }

        user_reading = reading_edges[reading_edges['user_id'] == user_id].copy()
        user_reading = user_reading[user_reading['item_id'].isin(valid_ids)]
        in_progress = set(
            user_reading.loc[
                user_reading['status'].astype(str).str.lower() == 'in progress',
                'item_id',
            ]
        )
        reading_all = set(user_reading['item_id'])

        user_dnr = dnr_edges[dnr_edges['user_id'] == user_id].copy()
        user_dnr = user_dnr[user_dnr['item_id'].isin(valid_ids)]
        dnr_all = set(user_dnr['item_id'])

        blocked_all = set(rating_map) | reading_all | dnr_all
        records[user_id] = {
            'rating_map': rating_map,
            'positive_ratings': positive_ratings,
            'in_progress': in_progress,
            'blocked_all': blocked_all,
        }

    return records


USER_RECORDS = build_user_records(
    ratings_edges=ratings_edges,
    reading_edges=reading_edges,
    dnr_edges=dnr_edges,
    valid_ids=graph_ids,
)

active_user_summary = pd.DataFrame(
    [
        {
            'user_id': user_id,
            'rated_items': len(record['rating_map']),
            'positive_ratings': len(record['positive_ratings']),
            'in_progress': len(record['in_progress']),
            'blocked_total': len(record['blocked_all']),
        }
        for user_id, record in USER_RECORDS.items()
    ]
).sort_values(['positive_ratings', 'rated_items'], ascending=[False, False])

ELIGIBLE_USERS = active_user_summary[active_user_summary['positive_ratings'] >= 3]['user_id'].tolist()

print('Eligible users for evaluation:', ELIGIBLE_USERS)
display(active_user_summary)


Eligible users for evaluation: ['isaacmeyer', 'chandler', 'bossitronio', 'avreylavelle', 'jnance142']


,user_id,rated_items,positive_ratings,in_progress,blocked_total
3,isaacmeyer,49,32,1,54
2,chandler,13,12,0,13
1,bossitronio,12,12,0,18
0,avreylavelle,10,8,0,25
4,jnance142,3,3,0,3


In [8]:
MANGA_V3_FRAME = manga_df.reset_index(drop=True).copy()


def build_seed_profile(seed_items, manga_lookup):
    genre_counter = Counter()
    theme_counter = Counter()

    for item_id in seed_items:
        if item_id not in manga_lookup.index:
            continue
        row = manga_lookup.loc[item_id]
        genre_counter.update(row['genres'])
        theme_counter.update(row['themes'])

    return {
        'age': None,
        'language': 'English',
        'preferred_genres': dict(genre_counter),
        'preferred_themes': dict(theme_counter),
        'blacklist_genres': {},
        'blacklist_themes': {},
    }


def rank_with_v3(candidate_items, seed_items, train_ratings, manga_lookup, manga_frame):
    if not candidate_items:
        return pd.DataFrame(columns=['id', 'v3_score'])

    profile = build_seed_profile(seed_items, manga_lookup)
    candidate_frame = manga_lookup.loc[list(candidate_items)].copy()

    ranked, _ = recommendation_scores(
        manga_df=manga_frame,
        profile=profile,
        current_genres=[],
        current_themes=[],
        read_manga=train_ratings,
        top_n=len(candidate_frame),
        mode='v3',
        earliest_year=None,
        prefiltered_df=candidate_frame,
    )

    if ranked is None or ranked.empty:
        return pd.DataFrame(columns=['id', 'v3_score'])

    return ranked[['id', 'combined_score']].rename(columns={'combined_score': 'v3_score'}).reset_index(drop=True)


def rank_with_popularity(candidate_items, manga_lookup):
    if not candidate_items:
        return pd.DataFrame(columns=['id', 'popularity_score'])

    ranked = (
        manga_lookup.loc[list(candidate_items), ['id', 'score', 'members']]
        .sort_values(['score', 'members'], ascending=[False, False])
        .rename(columns={'score': 'popularity_score'})
        .reset_index(drop=True)
    )
    return ranked


def build_eval_queries(
    user_records,
    universe_ids,
    n_splits=5,
    test_fraction=0.2,
    min_seed_items=2,
    seed=42,
):
    rng = random.Random(seed)
    queries = []

    for user_id, record in user_records.items():
        positives = sorted(record['positive_ratings'])
        if len(positives) < (min_seed_items + 1):
            continue

        max_test_size = len(positives) - min_seed_items
        test_size = max(1, int(round(len(positives) * test_fraction)))
        test_size = min(test_size, max_test_size)
        if test_size <= 0:
            continue

        for split_idx in range(n_splits):
            test_items = set(rng.sample(positives, k=test_size))
            seed_items = (set(positives) - test_items) | set(record['in_progress'])
            if len(seed_items) < min_seed_items:
                continue

            train_ratings = {
                item_id: rating
                for item_id, rating in record['rating_map'].items()
                if item_id not in test_items
            }
            excluded_items = set(record['blocked_all']) - test_items
            candidate_items = sorted(universe_ids - excluded_items)

            if not test_items.issubset(candidate_items):
                continue

            queries.append(
                {
                    'user_id': user_id,
                    'split_id': split_idx,
                    'seed_items': sorted(seed_items),
                    'test_items': sorted(test_items),
                    'candidate_items': candidate_items,
                    'train_ratings': train_ratings,
                }
            )

    return queries


def ranking_metrics(ranked_ids, relevant_ids, k=10):
    relevant_ids = set(relevant_ids)
    top_k = list(ranked_ids[:k])
    hits = [1 if item_id in relevant_ids else 0 for item_id in top_k]

    precision = sum(hits) / k if k else 0.0
    recall = sum(hits) / max(len(relevant_ids), 1)
    dcg = sum(hit / math.log2(idx + 2) for idx, hit in enumerate(hits))
    ideal_dcg = sum(1.0 / math.log2(idx + 2) for idx in range(min(len(relevant_ids), k)))
    ndcg = dcg / ideal_dcg if ideal_dcg else 0.0
    hitrate = 1.0 if any(hits) else 0.0

    mrr = 0.0
    for idx, item_id in enumerate(top_k):
        if item_id in relevant_ids:
            mrr = 1.0 / (idx + 1)
            break

    return {
        'precision_at_k': precision,
        'recall_at_k': recall,
        'ndcg_at_k': ndcg,
        'hitrate_at_k': hitrate,
        'mrr': mrr,
    }


## Train DeepWalk And Node2Vec

The defaults below are intentionally moderate so the notebook stays practical on a laptop or Raspberry Pi. They are easy to tune later if one method looks promising.


In [9]:
EMBEDDING_CONFIG = {
    'dimensions': 64,
    'walk_length': 20,
    'num_walks': 6,
    'window': 6,
    'workers': 1,
}
NODE2VEC_CONFIG = {
    'p': 1.0,
    'q': 2.0,
}

graph_nodes = [node for node, neighbors in adjacency.items() if neighbors]
print(f'Walkable nodes: {len(graph_nodes):,}')
print('Embedding config:', EMBEDDING_CONFIG)
print('Node2Vec config:', NODE2VEC_CONFIG)

started = time.time()
deepwalk_walks = generate_walks(
    nodes=graph_nodes,
    adjacency=adjacency,
    walk_length=EMBEDDING_CONFIG['walk_length'],
    num_walks=EMBEDDING_CONFIG['num_walks'],
    seed=RNG_SEED,
    method='deepwalk',
)
deepwalk_embeddings, deepwalk_backend = train_embeddings_from_walks(
    deepwalk_walks,
    dimensions=EMBEDDING_CONFIG['dimensions'],
    window=EMBEDDING_CONFIG['window'],
    seed=RNG_SEED,
    workers=EMBEDDING_CONFIG['workers'],
)
deepwalk_seconds = time.time() - started
del deepwalk_walks

started = time.time()
node2vec_walks = generate_walks(
    nodes=graph_nodes,
    adjacency=adjacency,
    walk_length=EMBEDDING_CONFIG['walk_length'],
    num_walks=EMBEDDING_CONFIG['num_walks'],
    seed=RNG_SEED,
    method='node2vec',
    adjacency_sets=adjacency_sets,
    p=NODE2VEC_CONFIG['p'],
    q=NODE2VEC_CONFIG['q'],
)
node2vec_embeddings, node2vec_backend = train_embeddings_from_walks(
    node2vec_walks,
    dimensions=EMBEDDING_CONFIG['dimensions'],
    window=EMBEDDING_CONFIG['window'],
    seed=RNG_SEED,
    workers=EMBEDDING_CONFIG['workers'],
)
node2vec_seconds = time.time() - started
del node2vec_walks

training_summary = pd.DataFrame(
    [
        {
            'model': 'DeepWalk',
            'embedding_count': len(deepwalk_embeddings),
            'backend': deepwalk_backend,
            'seconds': round(deepwalk_seconds, 2),
        },
        {
            'model': 'Node2Vec',
            'embedding_count': len(node2vec_embeddings),
            'backend': node2vec_backend,
            'seconds': round(node2vec_seconds, 2),
        },
    ]
)
display(training_summary)


Walkable nodes: 39,861
Embedding config: {'dimensions': 64, 'walk_length': 20, 'num_walks': 6, 'window': 6, 'workers': 1}
Node2Vec config: {'p': 1.0, 'q': 2.0}


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

,model,embedding_count,backend,seconds
0,DeepWalk,39861,gensim_word2vec,232.71
1,Node2Vec,39861,gensim_word2vec,258.64


## Compare Against v3

We now evaluate four models on the same repeated holdout splits:
- popularity baseline
- `v3` content-based recommender
- DeepWalk
- Node2Vec


In [10]:
EVAL_CONFIG = {
    'n_splits': 5,
    'test_fraction': 0.2,
    'min_seed_items': 2,
    'top_k': 10,
}

EVAL_QUERIES = build_eval_queries(
    user_records={user_id: USER_RECORDS[user_id] for user_id in ELIGIBLE_USERS},
    universe_ids=set(manga_df.index),
    n_splits=EVAL_CONFIG['n_splits'],
    test_fraction=EVAL_CONFIG['test_fraction'],
    min_seed_items=EVAL_CONFIG['min_seed_items'],
    seed=RNG_SEED,
)

print(f'Evaluation queries: {len(EVAL_QUERIES)}')
display(pd.DataFrame(EVAL_QUERIES).head(3))


def evaluate_models(queries, manga_lookup, manga_frame, deepwalk_embeddings, node2vec_embeddings, top_k=10):
    rows = []

    for query in queries:
        candidate_items = query['candidate_items']
        relevant_items = query['test_items']
        seed_items = query['seed_items']
        train_ratings = query['train_ratings']

        model_rankings = {
            'Popularity': rank_with_popularity(candidate_items, manga_lookup)['id'].tolist(),
            'V3': rank_with_v3(candidate_items, seed_items, train_ratings, manga_lookup, manga_frame)['id'].tolist(),
            'DeepWalk': score_candidates_from_embeddings(seed_items, candidate_items, deepwalk_embeddings)['id'].tolist(),
            'Node2Vec': score_candidates_from_embeddings(seed_items, candidate_items, node2vec_embeddings)['id'].tolist(),
        }

        for model_name, ranked_ids in model_rankings.items():
            metrics = ranking_metrics(ranked_ids, relevant_items, k=top_k)
            rows.append(
                {
                    'user_id': query['user_id'],
                    'split_id': query['split_id'],
                    'model': model_name,
                    **metrics,
                }
            )

    return pd.DataFrame(rows)


results_df = evaluate_models(
    queries=EVAL_QUERIES,
    manga_lookup=manga_df,
    manga_frame=MANGA_V3_FRAME,
    deepwalk_embeddings=deepwalk_embeddings,
    node2vec_embeddings=node2vec_embeddings,
    top_k=EVAL_CONFIG['top_k'],
)

summary_df = (
    results_df.groupby('model')
    .agg(
        queries=('model', 'size'),
        precision_at_10=('precision_at_k', 'mean'),
        recall_at_10=('recall_at_k', 'mean'),
        ndcg_at_10=('ndcg_at_k', 'mean'),
        hitrate_at_10=('hitrate_at_k', 'mean'),
        mrr=('mrr', 'mean'),
    )
    .sort_values(['ndcg_at_10', 'mrr', 'hitrate_at_10'], ascending=False)
)

display(summary_df.round(4))
display(results_df.head(12))


Evaluation queries: 25


,user_id,split_id,seed_items,test_items,candidate_items,train_ratings
0,isaacmeyer,0,"[1a051bb3-094e-4494-aa2e-fdac29b9ab5b, 237d527...","[192aa767-2479-42c1-9780-8d65a2efd36a, 478e692...","[00010753-b98c-4f53-b09e-d09a8a385f20, 0001183...","{'95a297a4-e8b5-4ac1-bcdc-d4ed160b8945': 2.0, ..."
1,isaacmeyer,1,"[192aa767-2479-42c1-9780-8d65a2efd36a, 1a051bb...","[246f2c0c-4c50-4ef4-b292-c17f0f68a17c, 5d1fc77...","[00010753-b98c-4f53-b09e-d09a8a385f20, 0001183...","{'95a297a4-e8b5-4ac1-bcdc-d4ed160b8945': 2.0, ..."
2,isaacmeyer,2,"[246f2c0c-4c50-4ef4-b292-c17f0f68a17c, 29ab698...","[192aa767-2479-42c1-9780-8d65a2efd36a, 1a051bb...","[00010753-b98c-4f53-b09e-d09a8a385f20, 0001183...","{'95a297a4-e8b5-4ac1-bcdc-d4ed160b8945': 2.0, ..."


,queries,precision_at_10,recall_at_10,ndcg_at_10,hitrate_at_10,mrr
model,,,,,,
Popularity,25,0.044,0.2200,0.1931,0.40,0.2767
V3,25,0.036,0.1133,0.1170,0.28,0.2267
Node2Vec,25,0.004,0.0067,0.0043,0.04,0.0067
DeepWalk,25,0.000,0.0000,0.0000,0.00,0.0000


,user_id,split_id,model,precision_at_k,recall_at_k,ndcg_at_k,hitrate_at_k,mrr
0,isaacmeyer,0,Popularity,0.2,0.333333,0.307983,1.0,0.5
1,isaacmeyer,0,V3,0.1,0.166667,0.302602,1.0,1.0
2,isaacmeyer,0,DeepWalk,0.0,0.000000,0.000000,0.0,0.0
3,isaacmeyer,0,Node2Vec,0.0,0.000000,0.000000,0.0,0.0
4,isaacmeyer,1,Popularity,0.1,0.166667,0.190921,1.0,0.5
5,isaacmeyer,1,V3,0.0,0.000000,0.000000,0.0,0.0
6,isaacmeyer,1,DeepWalk,0.0,0.000000,0.000000,0.0,0.0
7,isaacmeyer,1,Node2Vec,0.0,0.000000,0.000000,0.0,0.0
8,isaacmeyer,2,Popularity,0.0,0.000000,0.000000,0.0,0.0
9,isaacmeyer,2,V3,0.1,0.166667,0.302602,1.0,1.0


## Example Recommendations

The aggregate metrics tell us whether a method wins on holdout queries. This final section gives a quick qualitative preview for one real user so we can inspect the recommendation lists before deciding what to tune next.


In [11]:
def format_recommendation_frame(frame, score_column, model_name, manga_lookup, top_k=10):
    if frame.empty:
        return pd.DataFrame(columns=['model', 'id', 'title_name', 'english_name', score_column, 'score', 'members'])

    preview = frame.head(top_k).merge(
        manga_lookup[['id', 'title_name', 'english_name', 'score', 'members']],
        on='id',
        how='left',
    )
    preview.insert(0, 'model', model_name)
    return preview[['model', 'id', 'title_name', 'english_name', score_column, 'score', 'members']]


def preview_recommendations_for_user(user_id, manga_lookup, manga_frame, deepwalk_embeddings, node2vec_embeddings, top_k=10):
    user_key = str(user_id).strip().lower()
    record = USER_RECORDS[user_key]
    seed_items = sorted(record['positive_ratings'] | record['in_progress'])
    candidate_items = sorted(set(manga_lookup.index) - set(record['blocked_all']))

    popularity_frame = format_recommendation_frame(
        rank_with_popularity(candidate_items, manga_lookup),
        score_column='popularity_score',
        model_name='Popularity',
        manga_lookup=manga_lookup,
        top_k=top_k,
    )
    v3_frame = format_recommendation_frame(
        rank_with_v3(candidate_items, seed_items, record['rating_map'], manga_lookup, manga_frame),
        score_column='v3_score',
        model_name='V3',
        manga_lookup=manga_lookup,
        top_k=top_k,
    )
    deepwalk_frame = format_recommendation_frame(
        score_candidates_from_embeddings(seed_items, candidate_items, deepwalk_embeddings),
        score_column='graph_score',
        model_name='DeepWalk',
        manga_lookup=manga_lookup,
        top_k=top_k,
    )
    node2vec_frame = format_recommendation_frame(
        score_candidates_from_embeddings(seed_items, candidate_items, node2vec_embeddings),
        score_column='graph_score',
        model_name='Node2Vec',
        manga_lookup=manga_lookup,
        top_k=top_k,
    )

    return {
        'Popularity': popularity_frame,
        'V3': v3_frame,
        'DeepWalk': deepwalk_frame,
        'Node2Vec': node2vec_frame,
    }


preview_tables = preview_recommendations_for_user(
    EXAMPLE_USER_ID,
    manga_lookup=manga_df,
    manga_frame=MANGA_V3_FRAME,
    deepwalk_embeddings=deepwalk_embeddings,
    node2vec_embeddings=node2vec_embeddings,
    top_k=10,
)

for model_name, frame in preview_tables.items():
    print(f'\n### {model_name}')
    display(frame)


ValueError: 'id' is both an index level and a column label, which is ambiguous.

## Good Next Steps For This Notebook

1. If **Node2Vec** wins, tune `p` and `q` to test whether recommendation quality benefits more from local neighborhoods or more exploratory walks.
2. If **DeepWalk** and **Node2Vec** are close, tune the graph first rather than the walk model: especially author weight, theme weight, and common-tag cutoffs.
3. If both graph methods underperform `v3`, try a **hybrid score** such as `alpha * v3 + (1 - alpha) * graph_score`.
4. If coverage is the issue, try a richer graph that adds user-positive edges or author co-occurrence edges before moving to a heavier model.
5. If the graph methods look promising, the next stage should be either:
   - a tuned hybrid recommender, or
   - a heterogeneous graph representation experiment rather than a projected-only graph.
